In [ ]:
from __future__ import annotations

import pickle
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd


def _print_np_array_info(name: str, arr: np.ndarray, max_items: int = 5) -> None:
    """打印 numpy 数组的基础信息和前几个元素。"""
    print(f"[{name}] shape={arr.shape}, dtype={arr.dtype}, ndim={arr.ndim}")
    if arr.ndim == 0:
        print(f"  scalar value: {arr.item()}")
    elif arr.ndim == 1:
        print(f"  head({max_items}): {arr[:max_items]}")
    elif arr.ndim >= 2:
        print(f"  first row head({max_items}): {arr[0, :max_items]}")


def inspect_data_file(path: str, *, max_keys: int = 20, preview_rows: int = 5) -> Any:
    """
    统一读取并预览 .pkl / .npz 文件。

    返回：
        - pkl: 反序列化后的 Python 对象
        - npz: dict[str, np.ndarray]
    """
    file_path = Path(path)
    if not file_path.exists():
        raise FileNotFoundError(f"文件不存在: {file_path}")

    suffix = file_path.suffix.lower()
    print(f"File: {file_path}")

    if suffix in {".pkl", ".pickle"}:
        with open(file_path, "rb") as f:
            obj = pickle.load(f)

        print(f"Type: {type(obj)}")
        if isinstance(obj, dict):
            keys = list(obj.keys())
            print(f"dict size={len(keys)}, keys[:{max_keys}]={keys[:max_keys]}")
            if keys:
                k0 = keys[0]
                print(f"sample value type at key '{k0}': {type(obj[k0])}")
        elif isinstance(obj, (list, tuple)):
            print(f"len={len(obj)}")
            if len(obj) > 0:
                print(f"first item type={type(obj[0])}")
        elif isinstance(obj, np.ndarray):
            _print_np_array_info("pkl.ndarray", obj)

        return obj

    if suffix == ".npz":
        out: dict[str, np.ndarray] = {}
        with np.load(file_path, allow_pickle=True) as data:
            files = list(data.files)
            print(f"keys({len(files)}): {files[:max_keys]}")
            for k in files:
                arr = data[k]
                out[k] = arr
                _print_np_array_info(k, arr)

        # 尝试自动给出一个可 DataFrame 化的字段
        candidate = None
        for k, arr in out.items():
            if isinstance(arr, np.ndarray) and arr.ndim == 2:
                candidate = k
                break

        if candidate is not None:
            print(f"\nDataFrame preview from key='{candidate}'")
            display(pd.DataFrame(out[candidate]).head(preview_rows))
        else:
            print("\n未找到二维数组字段，跳过 DataFrame 预览。")

        return out

    raise ValueError(f"不支持的文件类型: {suffix}（仅支持 .pkl / .npz）")

In [2]:
# 示例1：查看 npz（你当前的 vbd_actions 文件）
npz_path = "/home/huangfukk/workspace/MAGAIL4AutoDrive/bc_baseline/outputs/vbd_actions/scenario_000000_vbd_actions.npz"
npz_obj = inspect_data_file(npz_path)

# 如果你想手动查看某个字段
# key = "vbd_actions"
# arr = npz_obj[key]
# print(arr.shape, arr.dtype)
# if arr.ndim == 2:
#     display(pd.DataFrame(arr).head())


# 示例2：查看 pkl（按需替换为你的场景文件）
# pkl_path = "/home/huangfukk/workspace/MAGAIL4AutoDrive/data/waymo_batches/part_0000_0002/part_0000_0002_0/sd_waymo_v1.2_1a8ce68ad3c7d0b6.pkl"
# pkl_obj = inspect_data_file(pkl_path)
#
# # pkl 若是 scenario dict，可进一步看 tracks 结构
# tracks = pkl_obj.get("tracks", {}) if isinstance(pkl_obj, dict) else {}
# print("num_tracks:", len(tracks) if isinstance(tracks, dict) else "N/A")
# if isinstance(tracks, dict) and tracks:
#     tid = next(iter(tracks))
#     print("sample track id:", tid)
#     print("sample track keys:", tracks[tid].keys())

['scenario_index', 'sdc_track_id', 'dt', 'vbd_actions', 'metadrive_actions', 'valid', 'speed_t', 'max_acc', 'max_steering', 'wheelbase', 'min_speed_for_steer', 'skipped', 'reason']
() int32


ValueError: Must pass 2-d input. shape=()